In [ ]:
import os
import re
import nltk
import networkx as nx
import numpy as np
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [ ]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

TXT_DIRECTORY = r"C:/Users/robin/OneDrive/Desktop/2_Career/1_Projects/2026_Quote_Extraction_from_Podcasts/Modern_Wisdom/TXT"
OUTPUT_DIRECTORY = r"C:/Users/robin/OneDrive/Desktop/2_Career/1_Projects/2026_Quote_Extraction_from_Podcasts/Modern_Wisdom/QUOTES"
TOP_N = 20
MIN_WORDS = 6
MAX_WORDS = 35

DANGLING_STARTS = {
    "it", "this", "that", "these", "those", "he", "she", "they",
    "and", "but", "so", "well", "yeah", "right", "because", "which"
}
HEDGE_WORDS = {
    "kind", "sort", "like", "maybe", "guess", "probably", "basically",
    "literally", "actually", "just", "really", "very", "quite", "pretty"
}

In [ ]:

model = SentenceTransformer("all-MiniLM-L6-v2")


def split_sentences(text):
    return [s.strip() for s in nltk.sent_tokenize(text) if s.strip()]


def passes_heuristics(sentence):
    words = sentence.split()
    if not (MIN_WORDS <= len(words) <= MAX_WORDS):
        return False
    first_word = re.sub(r"[^a-zA-Z]", "", words[0]).lower()
    if first_word in DANGLING_STARTS:
        return False
    if sentence.count('"') % 2 != 0:
        return False
    return True


def hedge_penalty(sentence):
    words = [w.lower().strip(".,!?") for w in sentence.split()]
    if not words:
        return 0
    hedges = sum(1 for w in words if w in HEDGE_WORDS)
    return hedges / len(words)


def rank_candidates(candidates, centrality_weight=0.5, distinctiveness_weight=0.35, hedge_weight=0.15):
    embeddings = model.encode(candidates, show_progress_bar=False, normalize_embeddings=True)
    sim_matrix = cosine_similarity(embeddings)
    np.fill_diagonal(sim_matrix, 0)

    graph = nx.from_numpy_array(sim_matrix)
    centrality = nx.pagerank(graph, max_iter=10000, tol=1e-4)

    avg_sim = sim_matrix.mean(axis=1)
    max_avg_sim = avg_sim.max() if avg_sim.max() > 0 else 1
    distinctiveness = 1 - (avg_sim / max_avg_sim)

    scored = []
    for i, s in enumerate(candidates):
        score = (
            centrality_weight * centrality[i]
            + distinctiveness_weight * distinctiveness[i]
            - hedge_weight * hedge_penalty(s)
        )
        scored.append((score, i, s))
    scored.sort(reverse=True)
    return scored, embeddings, sim_matrix


def mmr_select(scored, sim_matrix, top_n=TOP_N, diversity=0.7):
    if not scored:
        return []
    selected = [scored[0]]
    selected_idx = {scored[0][1]}
    remaining = scored[1:]

    while len(selected) < top_n and remaining:
        best = None
        best_mmr = -1e9
        for item in remaining:
            score, idx, s = item
            redundancy = max(sim_matrix[idx][j] for j in selected_idx)
            mmr = diversity * score - (1 - diversity) * redundancy
            if mmr > best_mmr:
                best_mmr = mmr
                best = item
        selected.append(best)
        selected_idx.add(best[1])
        remaining.remove(best)

    return selected


def process_file(txt_path, output_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()

    sentences = split_sentences(text)
    candidates = [s for s in sentences if passes_heuristics(s)]
    if not candidates:
        with open(output_path, "w", encoding="utf-8") as f:
            f.write("No candidate quotes found.\n")
        return

    scored, _, sim_matrix = rank_candidates(candidates)
    top_quotes = mmr_select(scored, sim_matrix, top_n=TOP_N)

    with open(output_path, "w", encoding="utf-8") as f:
        for rank, (score, idx, quote) in enumerate(top_quotes, 1):
            f.write(f"{rank}. (score: {score:.3f}) {quote}\n\n")


if __name__ == "__main__":
    os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)
    txt_files = [f for f in os.listdir(TXT_DIRECTORY) if f.lower().endswith(".txt")]

    for name in tqdm(txt_files, unit="episode"):
        in_path = os.path.join(TXT_DIRECTORY, name)
        out_path = os.path.join(OUTPUT_DIRECTORY, name.replace(".txt", "_quotes.txt"))
        process_file(in_path, out_path)

### Solution 1.2

In [ ]:
from google import genai
import json
import time 

client = genai.Client(api_key="")

def extract_quotes(chunk: str) -> list[dict]:
    prompt = f"""Extract any quotes attributed to a named person in this transcript excerpt.
The attribution phrase and the actual quote may be separated by other sentences.
Please only return the name of the person who stated the quote and return only the quote itself, 
without stating the exact words of the podcast guests or host introducing the quote.
Return ONLY a JSON list like: [{{"author of the quote": "...", "quote": "..."}}]
If none found, return [].

Text:
{chunk}"""

    resp = client.models.generate_content(model="gemini-3.1-flash-lite", contents=prompt)
    try:
        return json.loads(resp.text.strip("```json").strip("```"))
    except json.JSONDecodeError:
        return []

def chunk_text(text, size=2000, overlap=200):
    words = text.split()
    for i in range(0, len(words), size - overlap):
        yield " ".join(words[i:i+size])

all_quotes = []
for chunk in chunk_text(open('C:/Users/robin/OneDrive/Desktop/2_Career/1_Projects/2026_Quote_Extraction_from_Podcasts/Modern_Wisdom/TXT/example.txt', encoding="utf-8").read()):
    all_quotes.extend(extract_quotes(chunk))
    time.sleep(4.5) # ~13 requests/min, safely under the 15/min free-tier cap

In [10]:
all_quotes

[{'author of the quote': 'Rogan',
  'quote': "the worst thing that's ever happened to you is the worst thing that's ever happened to you."},
 {'author of the quote': 'Unknown',
  'quote': 'Number one, realize no one is coming to save you. Number two, take responsibility for your current position. Number three, be willing to sacrifice who you are for who you want to be.'},
 {'author of the quote': 'Neil Strauss',
  'quote': 'unspoken expectations are premeditated resentments'},
 {'author of the quote': 'Mark Manson',
  'quote': "Do hard shit. Not because it's fun, but because the win actually means something. You're blood for it. You broke for it. You earned it. Easy wins are forgettable. Hard ones change you. That's the point."},
 {'author of the quote': 'Gabe from I prevail',
  'quote': "You will always think you suck. That's okay. It's okay to suck compared to your standards as you grow."},
 {'author of the quote': 'Mark Manson',
  'quote': 'In sort of an era of AI because you can sp